In [ ]:
#01
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [4]:
#Configurations
BASE_DIR = 'D:/FYP/dental-vision/ML/data/processed'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VALID_DIR = os.path.join(BASE_DIR, 'valid')
TEST_DIR = os.path.join(BASE_DIR, 'test')
MODEL_DIR = 'D:/FYP/dental-vision/ML/Models'
os.makedirs(MODEL_DIR, exist_ok=True)


In [5]:
IMAGE_SIZE = (224, 224) #EfficientNetB0 default input size
BATCH_SIZE = 32
EPOCHS = 30 #Early stooping will cut this short if needed
NUM_CLASSES = 5 #Assuming 5 classes of dental conditions

In [6]:
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.21.0
GPU available: []


In [15]:
raw_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=42
)
class_names = raw_train.class_names
print('Class names:', class_names)

Found 13935 files belonging to 5 classes.
Class names: ['Bone_Loss', 'Caries', 'Impacted_Tooth', 'Missing_Teeth', 'Periapical_Lesion']


In [17]:
#save
with open(os.path.join(MODEL_DIR, 'class_names.json'), 'w') as f:
    json.dump(class_names, f)
print('Class names saved to class_names.json')


Class names saved to class_names.json


In [ ]:
#data augmentation applied only to training data -02
augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1), 
    layers.RandomContrast(0.1),
], name='augmentation')

In [29]:
def prepare(ds, augment=False):
    if augment:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    # Normalise always runs — whether augment is True or False
    ds = ds.map(lambda x, y: (x / 255.0, y),
                num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)

In [30]:
train_ds = prepare(raw_train, augment=True)
valid_ds = prepare(
    tf.keras.utils.image_dataset_from_directory(
        VALID_DIR,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        seed=42
    ),
)
test_ds = prepare(
    tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        seed=42
    )
)

Found 3970 files belonging to 5 classes.
Found 2094 files belonging to 5 classes.


In [31]:
print('Dataset ready.')
print(f'Training batches: {len(train_ds)}')
print(f'Validation batches: {len(valid_ds)}')
print(f'Test batches: {len(test_ds)}')

Dataset ready.
Training batches: 436
Validation batches: 125
Test batches: 66


In [32]:
#Load EfficientNetBO pre-trained on ImageNet
#include_top=False to exclude the final classification layer
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=IMAGE_SIZE + (3,)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [ ]:
#Freeze the base I am not retraining Google's feature extractor - 03
base_model.trainable = False

In [34]:
#Build the full model
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

In [36]:
model = tf.keras.Model(inputs, outputs, name='DentalVision')

In [37]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "DentalVision"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,416,168 (16.85 MB)

 Trainable params: 364,037 (1.39 MB)

 Non-trainable params: 4,052,131 (15.46 MB)